# Assignment 3 — Question 3

**Task:** Create Residue Interaction Graph (RIG) and Long-Range Interaction Network (LIN) models for **five** single-domain two-state folding proteins.

**Chosen proteins:** `1ubq`, `1hrc`, `1fkb`, `1aps`, `1csp`

**Procedure:**
1. Download PDB file from RCSB.
2. Extract Cα atomic coordinates.
3. Build RIG: connect residues *i* and *j* if Cα–Cα distance < **8 Å** and |i−j| ≥ 2.
4. Build LIN: connect residues *i* and *j* if Cα–Cα distance < **8 Å** and |i−j| > **12**.
5. Compute characteristic path length (L) and clustering coefficient (C) for each model.

**(a)** Tabular results for L and C.  
**(b)** Observations on topological properties and expected folding rate.

**Note on data:** PDB files for the five proteins are downloaded automatically from RCSB (https://files.rcsb.org) and saved to the `Question 3/pdb_files/` folder. If a download fails, manually download the `.pdb` file for the relevant protein from https://www.rcsb.org and place it in `Question 3/pdb_files/` using the filename format `<pdbid>.pdb` (e.g. `1ubq.pdb`).

In [ ]:
import os
import urllib.request
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from itertools import combinations

# Resolve this notebook's directory so all files are saved alongside it
try:
    SAVE_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    SAVE_DIR = os.path.abspath(os.path.dirname(''))
print(f'Save directory: {SAVE_DIR}')

## 1. Download PDB Files

In [ ]:
PROTEINS = ['1ubq', '1hrc', '1fkb', '1aps', '1csp']
PDB_DIR  = os.path.join(SAVE_DIR, 'pdb_files')
os.makedirs(PDB_DIR, exist_ok=True)

def download_pdb(pdb_id):
    pdb_id = pdb_id.lower()
    dest = os.path.join(PDB_DIR, f'{pdb_id}.pdb')
    if not os.path.exists(dest):
        url = f'https://files.rcsb.org/download/{pdb_id.upper()}.pdb'
        print(f'Downloading {pdb_id}...')
        urllib.request.urlretrieve(url, dest)
    else:
        print(f'{pdb_id}.pdb already exists.')
    return dest

pdb_paths = {pid: download_pdb(pid) for pid in PROTEINS}
print('All PDB files ready.')

## 2. Extract Cα Coordinates from PDB Files

We parse ATOM records manually, selecting only Cα atoms from the first chain of the first MODEL.

In [ ]:
def extract_ca_coords(pdb_file):
    """
    Parse a PDB file and extract Cα coordinates for the first chain of the first MODEL.
    Returns a list of (residue_number, np.array([x, y, z])).
    """
    ca_coords = []
    seen_residues = set()
    first_chain = None
    in_model = False

    with open(pdb_file, 'r') as f:
        for line in f:
            record = line[:6].strip()

            if record == 'MODEL':
                if in_model:          # second MODEL encountered → stop
                    break
                in_model = True
                continue

            if record == 'ENDMDL':
                break

            if record != 'ATOM':
                continue

            atom_name = line[12:16].strip()
            if atom_name != 'CA':
                continue

            chain_id = line[21]
            if first_chain is None:
                first_chain = chain_id
            if chain_id != first_chain:
                continue

            res_seq  = int(line[22:26].strip())
            ins_code = line[26].strip()
            res_key  = (res_seq, ins_code)

            if res_key in seen_residues:
                continue
            seen_residues.add(res_key)

            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            ca_coords.append((res_seq, np.array([x, y, z])))

    # Sort by residue number
    ca_coords.sort(key=lambda t: t[0])
    return ca_coords


ca_data = {}
for pid in PROTEINS:
    coords = extract_ca_coords(pdb_paths[pid])
    ca_data[pid] = coords
    print(f'{pid}: {len(coords)} residues')

## 3. Build RIG and LIN Models

**RIG:** Connect residues *i* and *j* if Cα–Cα distance < 8 Å **and** |i−j| ≥ 2 (exclude direct sequence neighbours to avoid trivial backbone contacts).  
**LIN:** Connect residues *i* and *j* if Cα–Cα distance < 8 Å **and** |i−j| > 12 (only long-range contacts).

In [ ]:
RIG_CUTOFF = 8.0   # Angstroms
LIN_SEQ_THRESHOLD = 12

def build_RIG(ca_coords, cutoff=RIG_CUTOFF):
    """Build Residue Interaction Graph (|i-j| >= 2, distance < cutoff)."""
    n = len(ca_coords)
    coords = np.array([c for _, c in ca_coords])
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(n):
        for j in range(i + 2, n):          # skip direct sequence neighbours
            dist = np.linalg.norm(coords[i] - coords[j])
            if dist < cutoff:
                G.add_edge(i, j, weight=dist)
    return G


def build_LIN(ca_coords, cutoff=RIG_CUTOFF, seq_threshold=LIN_SEQ_THRESHOLD):
    """Build Long-Range Interaction Network (|i-j| > seq_threshold, distance < cutoff)."""
    n = len(ca_coords)
    coords = np.array([c for _, c in ca_coords])
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(n):
        for j in range(i + seq_threshold + 1, n):   # |i-j| > 12
            dist = np.linalg.norm(coords[i] - coords[j])
            if dist < cutoff:
                G.add_edge(i, j, weight=dist)
    return G


rig_graphs = {}
lin_graphs = {}

for pid in PROTEINS:
    rig_graphs[pid] = build_RIG(ca_data[pid])
    lin_graphs[pid] = build_LIN(ca_data[pid])
    print(f'{pid} | RIG: {rig_graphs[pid].number_of_nodes()} nodes, '
          f'{rig_graphs[pid].number_of_edges()} edges  |  '
          f'LIN: {lin_graphs[pid].number_of_nodes()} nodes, '
          f'{lin_graphs[pid].number_of_edges()} edges')

## 4. Compute Characteristic Path Length (L) and Clustering Coefficient (C)

In [ ]:
def graph_metrics(G):
    """
    Compute characteristic path length L and average clustering coefficient C.
    L is computed on the largest connected component.
    """
    if G.number_of_edges() == 0:
        return float('nan'), float('nan')

    # Largest connected component
    lcc_nodes = max(nx.connected_components(G), key=len)
    G_lcc = G.subgraph(lcc_nodes).copy()

    L = nx.average_shortest_path_length(G_lcc)
    C = nx.average_clustering(G)
    return L, C


results = []
for pid in PROTEINS:
    L_rig, C_rig = graph_metrics(rig_graphs[pid])
    L_lin, C_lin = graph_metrics(lin_graphs[pid])
    n_res = len(ca_data[pid])
    results.append({
        'Protein':       pid.upper(),
        'N residues':    n_res,
        'RIG edges':     rig_graphs[pid].number_of_edges(),
        'RIG L':         round(L_rig, 4),
        'RIG C':         round(C_rig, 4),
        'LIN edges':     lin_graphs[pid].number_of_edges(),
        'LIN L':         round(L_lin, 4),
        'LIN C':         round(C_lin, 4),
    })
    print(f'{pid.upper()} done.')

df = pd.DataFrame(results)
print(df.to_string(index=False))

## 5. (a) Results Table

In [ ]:
print('\n=== Characteristic Path Length (L) and Clustering Coefficient (C) ===')
print(df.to_string(index=False))

# Nicely formatted table
try:
    from IPython.display import display
    display(df.style.set_caption('RIG and LIN metrics for five proteins'))
except ImportError:
    pass

## 6. Visualise Contact Maps

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8))

for col, pid in enumerate(PROTEINS):
    n = len(ca_data[pid])

    for row, (G, label) in enumerate([(rig_graphs[pid], 'RIG'), (lin_graphs[pid], 'LIN')]):
        mat = np.zeros((n, n))
        for (i, j) in G.edges():
            mat[i][j] = 1
            mat[j][i] = 1
        axes[row][col].imshow(mat, cmap='Blues', interpolation='none')
        axes[row][col].set_title(f'{pid.upper()} — {label}', fontsize=9)
        axes[row][col].set_xlabel('Residue', fontsize=7)
        axes[row][col].set_ylabel('Residue', fontsize=7)

plt.suptitle('Contact Maps: RIG (top row) and LIN (bottom row)', fontsize=12)
plt.tight_layout()
out_path = os.path.join(SAVE_DIR, 'contact_maps.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 7. (b) Observations

### Topological Properties

**RIG:**
- Includes both short- and long-range contacts. All five proteins show small-world character: moderate path lengths L and high clustering coefficients C relative to random graphs of the same size and density.
- Small proteins (1CSP, 1UBQ) have fewer residues and denser relative contact maps, leading to lower L values.
- Larger proteins (1FKB, 1HRC) have more residues, so L increases, while C stays elevated due to local structural packing.

**LIN (long-range only, |i-j| > 12):**
- LIN removes all short- and medium-range contacts, leaving only contacts between residues far apart in sequence but close in 3D space — typically secondary structure interfaces and β-sheet pairings.
- LIN networks are sparser and often disconnected or near-disconnected for small proteins, which inflates L and reduces C.
- Proteins with more extensive β-sheet structure (e.g., 1FKB, 1APS) retain more LIN edges and show better connectivity.

### Expected Rate of Folding

- Proteins with a higher fraction of long-range contacts (higher LIN density relative to total RIG contacts) tend to fold more slowly, because forming long-range contacts requires a larger conformational search.
- Proteins dominated by local contacts (most edges in RIG but few in LIN) — such as mostly-helical proteins — can nucleate folding quickly.
- The contact order (CO) — closely related to the fraction of long-range contacts — is an established predictor of folding rate: high CO → slower folding.

### Summary Table (qualitative)

| Protein | Structure type | LIN/RIG ratio (approx.) | Expected folding speed |
|---------|---------------|------------------------|------------------------|
| 1UBQ    | Mixed α/β     | moderate               | moderate               |
| 1HRC    | Mostly α      | low                    | fast                   |
| 1FKB    | β-sheet rich  | higher                 | slower                 |
| 1APS    | Mixed α/β     | moderate               | moderate               |
| 1CSP    | β-barrel      | higher                 | slower                 |